# TMDB data pull

Pulls the ten most-voted films for every year TMDB covers, then every review attached to
those films. Both are saved as separate files.

Needs `TMDB_API_KEY` in a `.env` file in this folder.

In [1]:
import numpy as np
import pandas as pd
import requests
import json
import os
import time
import dotenv

In [2]:
mybot = 'ds6021'
botversion = '0.0'
botwebsite = 'Data access for a class assignment'
email = 'conorkstewart@gmail.com'
software = 'python-requests'
softwareversion = requests.__version__

useragent = f'{mybot}/{botversion} ({botwebsite}; {email}) {software}/{softwareversion}'

## Credential

Read from `.env`. The length check confirms it loaded without printing the value.

In [3]:
dotenv.load_dotenv()
api_key = os.getenv("TMDB_API_KEY")
len(api_key)

32

In [4]:
root = "https://api.themoviedb.org/3"
headers = {"User-Agent": useragent}
params = {"api_key": api_key, "append_to_response": "credits,keywords,external_ids,release_dates"}

## One request

Sanity check before the loops.

In [5]:
r = requests.get(root + "/movie/19995", params=params, headers=headers)
r

<Response [200]>

In [6]:
data = r.json()
pd.Series({field: data.get(field) for field in
           ["title", "release_date", "runtime", "budget",
            "revenue", "vote_count", "vote_average"]})

title               Avatar
release_date    2009-12-16
runtime                162
budget           237000000
revenue         2923706026
vote_count           34966
vote_average         7.611
dtype: object

## Picking the movies

`discover` takes a release-date window and a sort order, so one request per year returns
that year's films ordered by how many votes they have.

`picks` is the single source of truth for which films the rest of the notebook uses: both
the detail pull and the review pull read from it, so changing `years` or the sort changes
everything downstream.

In [7]:
years = range(1874, 2027)

picks = []

for year in years:
    params_year = {
        "api_key": api_key,
        "sort_by": "vote_count.desc",
        "primary_release_date.gte": f"{year}-01-01",
        "primary_release_date.lte": f"{year}-12-31",
        "page": 1,
    }
    r = requests.get(root + "/discover/movie", params=params_year, headers=headers)
    for movie in r.json().get("results", [])[:10]:
        picks.append({
            "tmdb_id": movie["id"],
            "title": movie["title"],
            "release_date": movie["release_date"],
            "vote_count": movie["vote_count"],
        })
    time.sleep(0.01)

picks = pd.DataFrame(picks)
picks.shape

(1364, 4)

In [8]:
picks.groupby(picks.release_date.str[:4]).size()

release_date
1874     1
1887     1
1888     5
1889     3
1890     5
        ..
2022    10
2023    10
2024    10
2025    10
2026    10
Length: 141, dtype: int64

## Full records

One detail request per picked film.

In [9]:
rows = []

for number, pick in enumerate(picks.itertuples(), start=1):
    r = requests.get(root + f"/movie/{pick.tmdb_id}", params=params, headers=headers)
    d = r.json()
    crew = d.get("credits", {}).get("crew", [])
    cast = d.get("credits", {}).get("cast", [])
    rows.append({
        "tmdb_id": pick.tmdb_id,
        "imdb_id": d.get("external_ids", {}).get("imdb_id"),
        "title": d.get("title"),
        "release_date": d.get("release_date"),
        "runtime": d.get("runtime"),
        "budget": d.get("budget"),
        "revenue": d.get("revenue"),
        "vote_average": d.get("vote_average"),
        "vote_count": d.get("vote_count"),
        "original_language": d.get("original_language"),
        "genres": "|".join(g["name"] for g in d.get("genres", [])),
        "companies": "|".join(c["name"] for c in d.get("production_companies", [])),
        "countries": "|".join(c["name"] for c in d.get("production_countries", [])),
        "collection": (d.get("belongs_to_collection") or {}).get("name"),
        "cast_count": len(cast),
        "crew_count": len(crew),
        "vfx_crew": sum(1 for c in crew if c.get("department") == "Visual Effects"),
    })
    if number % 100 == 0:
        print(f"{number} of {len(picks)}")
    elif number == len(picks):
        print(f"{number} of {len(picks)}")
    time.sleep(0.01)

movies = pd.DataFrame(rows)
movies.shape

100 of 1364
200 of 1364
300 of 1364
400 of 1364
500 of 1364
600 of 1364
700 of 1364
800 of 1364
900 of 1364
1000 of 1364
1100 of 1364
1200 of 1364
1300 of 1364
1364 of 1364


(1364, 17)

## Save the movie pull

In [10]:
os.makedirs("data", exist_ok=True)
movies.to_csv("data/tmdb_top10_per_year.csv", index=False)
os.path.getsize("data/tmdb_top10_per_year.csv")

229974

## Reviews

The reviews endpoint pages 20 at a time, so a film with 60 reviews needs three requests.
The inner loop keeps going until it reaches `total_pages`.

Films with no reviews return an empty list and contribute no rows.

In [11]:
reviews = []

for number, pick in enumerate(picks.itertuples(), start=1):
    page = 1
    while True:
        params_page = {"api_key": api_key, "page": page}
        r = requests.get(root + f"/movie/{pick.tmdb_id}/reviews",
                         params=params_page, headers=headers)
        d = r.json()
        for review in d.get("results", []):
            details = review.get("author_details") or {}
            reviews.append({
                "tmdb_id": pick.tmdb_id,
                "title": pick.title,
                "review_id": review.get("id"),
                "author": review.get("author"),
                "rating": details.get("rating"),
                "created_at": review.get("created_at"),
                "content": review.get("content"),
            })
        if page >= d.get("total_pages", 1):
            break
        page += 1
        time.sleep(0.01)
    if number % 100 == 0:
        print(f"{number} of {len(picks)}")
    elif number == len(picks):
        print(f"{number} of {len(picks)}")
    time.sleep(0.01)

reviews = pd.DataFrame(reviews)
reviews.shape

100 of 1364
200 of 1364
300 of 1364
400 of 1364
500 of 1364
600 of 1364
700 of 1364
800 of 1364
900 of 1364
1000 of 1364
1100 of 1364
1200 of 1364
1300 of 1364
1364 of 1364


(4982, 7)

In [12]:
pd.Series({
    "films pulled": len(picks),
    "films with at least one review": reviews.tmdb_id.nunique(),
    "reviews collected": len(reviews),
    "median characters per review": int(reviews.content.str.len().median()),
})

films pulled                      1364
films with at least one review     963
reviews collected                 4982
median characters per review       948
dtype: int64

## Save the review pull

Separate file from the movie data, joined on `tmdb_id`.

In [13]:
reviews.to_csv("data/tmdb_reviews.csv", index=False)
os.path.getsize("data/tmdb_reviews.csv")

6925368